In [5]:
import sys
import time
import statistics
import pandas as pd
import torch
from pathlib import Path

# 1. Умное определение корня проекта и приоритетный импорт (insert)
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT))
print(f"📂 Корень проекта: {PROJECT_ROOT}")

from src.model import build_model
from src.dataset import read_train_csv, filter_existing_rows, TrainDataset, stratified_split
from src.metrics import AICMeter

# 2. Строгая воспроизводимость всех генераторов (из утилит)
from src.utils import set_seed
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Тестируем архитектуры на: {device}")

# 3. БОЕВОЙ размер для честного замера
IMG_SIZE = 256

📂 Корень проекта: d:\projects\digital-detective
🚀 Тестируем архитектуры на: cpu


In [6]:
def benchmark_encoder(encoder_name: str, img_size: int):
    # Изолированный импорт для совместимости с PyTorch < 2.1
    try:
        from torch.utils.flop_counter import FlopCounterMode
    except ImportError:
        print(f"⚠️ FlopCounterMode недоступен. Пропуск замера FLOPs.")
        FlopCounterMode = None

    model = build_model(
        architecture='unet', 
        encoder_name=encoder_name, 
        encoder_weights=None
    ).to(device).eval()
    
    dummy_input = torch.randn(1, 3, img_size, img_size, device=device)

    # 1. Замер FLOPs
    gflops = None
    if FlopCounterMode is not None:
        try:
            flop_ctx = FlopCounterMode(display=False)
        except TypeError:
            flop_ctx = FlopCounterMode(model, display=False)

        with flop_ctx as fc, torch.no_grad():
            model(dummy_input)
            
        total_flops = fc.get_total_flops()
        if total_flops > 0:
            gflops = total_flops / 1e9

    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    # 2. Замер Latency
    warmup_iters = 20
    iters = 100
    latencies_ms = []

    with torch.inference_mode():
        for _ in range(warmup_iters):
            model(dummy_input)
        if device.type == 'cuda': torch.cuda.synchronize(device)

        for _ in range(iters):
            if device.type == 'cuda': torch.cuda.synchronize(device)
            t0 = time.perf_counter()
            model(dummy_input)
            if device.type == 'cuda': torch.cuda.synchronize(device)
            latencies_ms.append((time.perf_counter() - t0) * 1000)

    med_latency = statistics.median(latencies_ms)

    del model, dummy_input
    if device.type == 'cuda': torch.cuda.empty_cache()

    # Оценка лимитов
    pass_limits = "❌ Нет"
    if gflops is None:
        pass_limits = "❓ Без FLOPs"
    elif gflops <= 100 and med_latency <= 50:
        pass_limits = "✅ Да"

    return {
        "Encoder": encoder_name,
        "Params (M)": round(params_m, 2),
        "GFLOPs": round(gflops, 2) if gflops else None,
        "Latency Med (ms)": round(med_latency, 2),
        "Pass Limits?": pass_limits
    }

In [7]:
data_dir = PROJECT_ROOT / 'data'
rows = read_train_csv(data_dir / 'stage1' / 'train.csv')
rows = filter_existing_rows(rows, data_dir, show_progress=False)

# ВАЖНО: должно совпадать с --val-split из run_train.py (по умолчанию 0.1)
VAL_RATIO = 0.1
train_rows, val_rows = stratified_split(rows, val_ratio=VAL_RATIO, seed=42)

# Берем 500 примеров ИЗ ВАЛИДАЦИИ
val_subset = val_rows[:500]
print(f"📊 Валидация: {len(val_subset)} из {len(val_rows)}")

val_ds = TrainDataset(val_subset, img_size=IMG_SIZE, train=False, data_dir=data_dir)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

def quick_eval(encoder_name, max_batches=20):
    model = build_model(
        architecture='unet',
        encoder_name=encoder_name,
        encoder_weights='imagenet',
    ).to(device).eval()
    
    meter = AICMeter(pred_threshold=0.5, gt_threshold=0.5)
    
    with torch.inference_mode():
        for batch_idx, batch in enumerate(val_loader):
            if batch_idx >= max_batches:
                break
            
            images = batch['image'].to(device)
            masks = batch['mask'].to(device).float()
            
            logits = model(images)
            probs = torch.sigmoid(logits)
            meter.update(probs, masks)
    
    aic = meter.compute()
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()
    
    return aic

📊 Валидация: 500 из 10369


In [ ]:
encoders_to_test = [
    'resnet18',
    'resnet34',
    'efficientnet-b0',
    'efficientnet-b2',
    'mobilenet_v2'
]

results = []

print("Начинаем профилирование и замер качества...")
for enc in encoders_to_test:
    print(f"Тестируем {enc}...")
    try:
        # Без лишних проверок, всё линейно
        res = benchmark_encoder(enc, img_size=IMG_SIZE)
        aic = quick_eval(enc, max_batches=25)
        
        res["Val AIC (fast)"] = round(aic, 4)
        results.append(res)
    except Exception as e:
        print(f"  ❌ Ошибка с {enc}: {e}")
        results.append({
            "Encoder": enc,
            "Params (M)": None,
            "GFLOPs": None,
            "Latency Med (ms)": None,
            "Pass Limits?": f"❌ Ошибка: {str(e)[:60]}...", # 60 символов для контекста
            "Val AIC (fast)": None,
        })

df_results = pd.DataFrame(results)

# Сортировка: сначала прошедшие лимиты, затем лучшие по качеству (AIC)
df_results = df_results.sort_values(
    by=["Pass Limits?", "Val AIC (fast)"], 
    ascending=[False, False]
)
display(df_results)

Начинаем профилирование и замер качества...
Тестируем resnet18...


INFO: HTTP Request: HEAD https://huggingface.co/smp-hub/resnet18.imagenet/resolve/3f2325ff978283d47aa6a1d6878ca20565622683/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/smp-hub/resnet18.imagenet/3f2325ff978283d47aa6a1d6878ca20565622683/config.json?%2Fsmp-hub%2Fresnet18.imagenet%2Fresolve%2F3f2325ff978283d47aa6a1d6878ca20565622683%2Fconfig.json=&etag=%22879c59b3870194ae248b52920479fed460415e62%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/smp-hub/resnet18.imagenet/3f2325ff978283d47aa6a1d6878ca20565622683/config.json?%2Fsmp-hub%2Fresnet18.imagenet%2Fresolve%2F3f2325ff978283d47aa6a1d6878ca20565622683%2Fconfig.json=&etag=%22879c59b3870194ae248b52920479fed460415e62%22 "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

INFO: HTTP Request: HEAD https://huggingface.co/smp-hub/resnet18.imagenet/resolve/3f2325ff978283d47aa6a1d6878ca20565622683/model.safetensors "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/smp-hub/resnet18.imagenet/xet-read-token/3f2325ff978283d47aa6a1d6878ca20565622683 "HTTP/1.1 200 OK"


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

...

INFO: HTTP Request: HEAD https://huggingface.co/smp-hub/efficientnet-b0.imagenet/resolve/1bbe7ecc1d5ea1d2058de1a2db063b8701aff314/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/smp-hub/efficientnet-b0.imagenet/1bbe7ecc1d5ea1d2058de1a2db063b8701aff314/config.json?%2Fsmp-hub%2Fefficientnet-b0.imagenet%2Fresolve%2F1bbe7ecc1d5ea1d2058de1a2db063b8701aff314%2Fconfig.json=&etag=%226e9d307729de5f4e5960c8adebf75c77a0774497%22 "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/smp-hub/efficientnet-b0.imagenet/1bbe7ecc1d5ea1d2058de1a2db063b8701aff314/config.json?%2Fsmp-hub%2Fefficientnet-b0.imagenet%2Fresolve%2F1bbe7ecc1d5ea1d2058de1a2db063b8701aff314%2Fconfig.json=&etag=%226e9d307729de5f4e5960c8adebf75c77a0774497%22 "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

INFO: HTTP Request: HEAD https://huggingface.co/smp-hub/efficientnet-b0.imagenet/resolve/1bbe7ecc1d5ea1d2058de1a2db063b8701aff314/model.safetensors "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/smp-hub/efficientnet-b0.imagenet/xet-read-token/1bbe7ecc1d5ea1d2058de1a2db063b8701aff314 "HTTP/1.1 200 OK"


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

### Вывод по экспериментам:
Проверены аппаратные лимиты (<=100 GFLOPs, <=50ms) на боевом разрешении (256x256) и базовая экстракция признаков на отложенной валидационной выборке. Утечки данных исключены (использован `stratified_split`).

Для финального обучения выбран **[ВСТАВИТЬ ПОБЕДИТЕЛЯ]**, так как он проходит аппаратный контроль и показывает наивысший AIC из коробки.